In [1]:
import polars as pl
from dotenv import load_dotenv
import os
import pandas as pd

In [10]:
def features(df):
    
    property_mapping = {'rental unit': 'home/apt',
        'condo': 'home/apt',
        'home': 'home/apt',
        'loft': 'home/apt',
        'townhouse': 'home/apt',
        'serviced apartment': 'home/apt',
        'guest suite': 'home/apt',
        'guesthouse': 'home/apt',
        'villa': 'home/apt',
        'tiny home': 'home/apt',
        'place': 'home/apt',
        'casa particular': 'home/apt',
        'private room': 'home/apt',
        'vacation home': 'home/apt',
        'bed and breakfast': 'hotel',
        'hotel': 'hotel',
        'boutique hotel': 'hotel',
        'hostel': 'hotel',
        'aparthotel': 'hotel',
        'heritage hotel': 'hotel'
    }
    
    amenities_rename = {
        'central air conditioning': 'air conditioning',
        'private backyard \\u2013 fully fenced': 'backyard',
        'private patio or balcony': 'balcony',
        'patio or balcony': 'balcony'
    }
    amenities_keep = {
        'dedicated workspace',
        'tv',
        'heating',
        'private entrance',
        'outdoor dining area',
        'self check-in',
        'canal view',
        'cleaning available during stay',
        'free parking on premises',
        'elevator',
        'gym',
        'city skyline view',
        'ev charger',
        'backyard',
        'air conditioning',
        'balcony'
    }
    
    df = df.with_columns(
        # Missing indicators
        pl.col('host_is_superhost')
        .is_null()
        .cast(pl.UInt8)
        .alias('host_is_superhost_missing'),
        
        pl.col('instant_bookable')
        .is_null()
        .cast(pl.UInt8)
        .alias('instant_bookable_missing'),
        
        pl.col('license')
        .is_not_null()
        .cast(pl.UInt8)
        .alias('has_license'),
        
        # Binary encoding
        pl.col('bathroom_shared')
        .cast(pl.UInt8)
        .alias('bathroom_shared'),
        
        pl.col('host_is_superhost')
        .replace_strict({"t": 1, "f": 0}, default=0)
        .alias('host_is_superhost'),
        
        pl.col('instant_bookable')
        .replace_strict({"t": 1, "f": 0}, default=0)
        .alias('instant_bookable'),
        
        pl.col('amenities')
        .str.strip_chars('[]')
        .str.replace_all('"', '')
        .str.split(',')
        .list.len()
        .alias('amenities_count'),
        
        pl.col('beds')
        .fill_null(pl.col('accommodates')),
        
        pl.col('bedrooms')
        .fill_null(pl.col('accommodates')),
        
        (pl.col('snapshot_date')
        .cast(pl.Date) - pl.col('first_review')
        .cast(pl.Date))
        .dt.total_days()
        .alias('days_since_first_review'),
        
        (pl.col('snapshot_date')
        .cast(pl.Date) - pl.col('last_review')
        .cast(pl.Date))
        .dt.total_days()
        .alias('days_since_last_review'),
        
        pl.col('first_review')
        .is_null()
        .cast(pl.UInt8)
        .alias('has_no_review'),
        
        pl.col('review_scores_rating')
        .fill_null(pl.col('review_scores_rating').mean()),
        
        pl.col('review_scores_cleanliness')
        .fill_null(pl.col('review_scores_cleanliness').mean()),
        
        pl.col('review_scores_location')
        .fill_null(pl.col('review_scores_location').mean()),
        
        pl.col('reviews_per_month')
        .fill_null(0),
        
        pl.col('price')
        .log()
        .alias('log_price')
    )
    
    df = df.with_columns(
        pl.col('days_since_first_review')
        .fill_null(99999),
        
        pl.col('days_since_last_review')
        .fill_null(99999)
    )
    
    # Remove redundant wording in property_type
    regex = r"^(Entire |Private room in |Shared room in |Room in )"
    df = df.with_columns(
        pl.col('property_type')
        .str.replace(regex, '')
        .str.to_lowercase()
    )
    
    # Map the property_type to a category
    df = df.with_columns(pl.col('property_type').replace_strict(property_mapping, default='unique'))

    # Create dummies for neccessary columns (One hot encoding)
    property_dummies = (
        df.select('property_type')
        .to_dummies()
    )
    property_dummies = property_dummies.drop('property_type_home/apt')
    
    neighbourhood_dummies = (
        df.select('neighbourhood_cleansed')
        .to_dummies()
        .rename(lambda x: x.replace('neighbourhood_cleansed_', 'neighbourhood_'))
    )
    neighbourhood_dummies = neighbourhood_dummies.drop('neighbourhood_Centrum-West')
    
    room_type_dummies = (
        df.select('room_type')
        .to_dummies()
    )
    room_type_dummies = room_type_dummies.drop('room_type_Entire home/apt')
    
    # Clean and and turn amenities into list
    df = df.with_columns(
        pl.col('amenities')
        .str.strip_chars('[]')
        .str.replace_all('"', '')
        .str.split(',')
        .list.eval(
            pl.element()
            .str.strip_chars_start()
            )
        .alias('amenities')
    )
    
    # Keep only neccesary entries from amenities
    df = df.with_columns(
        pl.col('amenities')
        .list.eval(
            pl.element()
            .str.to_lowercase()
            .replace(amenities_rename)
            )
        .list.set_intersection(amenities_keep)
        .alias('amenities_clean')
    )
    
    # Store expressions to optimize / One hot encoding amenities
    expressions = []
    for i in amenities_keep:
        expressions.append(
            pl.col('amenities_clean')
            .list.contains(i)
            .cast(pl.UInt8)
            .alias(f'has_{i}'.replace(' ', '_'))
            )
        
    df = df.with_columns(expressions)
    
    # Add the dummies at the end to the dataframe
    df = pl.concat(
        [df, room_type_dummies, neighbourhood_dummies, property_dummies],
        how='horizontal_extend'
    )
    
    # Drop the un-encoded columns
    df = df.drop(['room_type', 'neighbourhood_cleansed', 'property_type', 'license', 'amenities', 'amenities_clean'])
    
    # Filter
    df = df.filter(
        (pl.col('minimum_nights').is_not_null()) & (pl.col('bathrooms_final').is_not_null()) & (pl.col('latitude').is_not_null()) & (pl.col('longitude').is_not_null())
    )
    # Rename columns
    df = df.rename(lambda col: (col.lower().replace(" ", "_")))
    
    return df

In [14]:
load_dotenv()
LISTINGS_DATA = os.getenv('CONCAT_LISTINGS_PATH')
CALENDER_DATA = os.getenv('CONCAT_CALENDER_PATH')
PRED_DATA = os.getenv('PRED_PATH')

ldf = pl.read_parquet(LISTINGS_DATA)
predictions = pl.read_parquet(PRED_DATA)

In [15]:
ldf = features(ldf)

In [16]:
amenities_keep = [
    'city_skyline_view',
    'heating',
    'dedicated_workspace',
    'elevator',
    'ev_charger',
    'gym',
    'canal_view',
    'outdoor_dining_area',
    'private_entrance',
    'tv',
    'cleaning_available_during_stay',
    'balcony',
    'self_check-in',
    'backyard',
    'free_parking_on_premises',
    'air_conditioning'
]

neighbourhoods_keep = [
    'ijburg_-_zeeburgereiland',
    'noord-oost',
    'de_baarsjes_-_oud-west',
    'gaasperdam_-_driemond',
    'bos_en_lommer',
    'bijlmer-centrum',
    'slotervaart',
    'oud-oost',
    'bijlmer-oost',
    'watergraafsmeer',
    'noord-west',
    'de_aker_-_nieuw_sloten',
    'zuid',
    'centrum-oost',
    'geuzenveld_-_slotermeer',
    'osdorp',
    'westerpark',
    'de_pijp_-_rivierenbuurt',
    'oostelijk_havengebied_-_indische_buurt',
    'oud-noord',
    'buitenveldert_-_zuidas'
]

FEATURES = [
    'host_is_superhost',
    'accommodates',
    'beds',
    'bedrooms',
    'number_of_reviews',
    'review_scores_rating',
    'review_scores_cleanliness',
    'review_scores_location',
    'instant_bookable',
    'reviews_per_month',
    'bathroom_shared',
    'bathrooms_final',
    'host_is_superhost_missing',
    'has_license',
    'instant_bookable_missing',
    'amenities_count',
    'days_since_first_review',
    'days_since_last_review',
    'has_no_review',
    'room_type_hotel_room',
    'room_type_private_room',
    'room_type_shared_room',
    'property_type_hotel',
    'property_type_unique'
]

for i in amenities_keep:
    FEATURES.append(f'has_{i}')
for i in neighbourhoods_keep:
    FEATURES.append(f'neighbourhood_{i}')


In [17]:
ldf.shape

(11623, 139)

In [18]:
df = ldf.join(predictions, on=['id', 'snapshot_date'], how='left')

In [19]:
df.shape

(11623, 140)

In [ ]:
df

host_is_superhost,accommodates,beds,bedrooms,number_of_reviews,review_scores_rating,review_scores_cleanliness,review_scores_location,instant_bookable,reviews_per_month,bathroom_shared,bathrooms_final,host_is_superhost_missing,has_license,instant_bookable_missing,amenities_count,days_since_first_review,days_since_last_review,has_no_review,room_type_hotel_room,room_type_private_room,room_type_shared_room,property_type_hotel,property_type_unique,has_city_skyline_view,has_heating,has_dedicated_workspace,has_elevator,has_ev_charger,has_gym,has_canal_view,has_outdoor_dining_area,has_private_entrance,has_tv,has_cleaning_available_during_stay,has_balcony,has_self_check-in,has_backyard,has_free_parking_on_premises,has_air_conditioning,neighbourhood_ijburg_-_zeeburgereiland,neighbourhood_noord-oost,neighbourhood_de_baarsjes_-_oud-west,neighbourhood_gaasperdam_-_driemond,neighbourhood_bos_en_lommer,neighbourhood_bijlmer-centrum,neighbourhood_slotervaart,neighbourhood_oud-oost,neighbourhood_bijlmer-oost,neighbourhood_watergraafsmeer,neighbourhood_noord-west,neighbourhood_de_aker_-_nieuw_sloten,neighbourhood_zuid,neighbourhood_centrum-oost,neighbourhood_geuzenveld_-_slotermeer,neighbourhood_osdorp,neighbourhood_westerpark,neighbourhood_de_pijp_-_rivierenbuurt,neighbourhood_oostelijk_havengebied_-_indische_buurt,neighbourhood_oud-noord,neighbourhood_buitenveldert_-_zuidas
i64,i64,f64,f64,i64,f64,f64,f64,i64,f64,u8,f64,u8,u8,u8,u32,i64,i64,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8
1,3,2.0,2.0,656,4.94,4.94,4.98,0,3.45,0,1.0,0,1,1,52,5708,8,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
0,4,4.0,2.0,187,4.86,4.82,4.67,0,0.98,0,1.5,0,1,1,58,5724,11,0,0,0,0,0,0,0,1,1,1,1,0,0,1,1,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0
1,2,1.0,1.0,631,4.86,4.82,4.93,0,3.35,0,1.0,0,1,1,20,5641,15,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,2,1.0,2.0,334,4.8,4.66,4.9,0,1.84,0,1.0,0,1,1,28,5438,2,0,0,1,0,0,1,0,1,0,0,0,0,1,0,0,0,1,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0
0,2,2.0,2.0,447,4.69,4.88,4.48,0,2.46,1,1.0,0,1,1,15,5456,22,0,0,1,0,1,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
0,1,1.0,1.0,0,4.838775,4.784995,4.826571,0,0.0,0,1.0,0,0,0,1,99999,99999,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0
1,2,1.0,1.0,0,4.838775,4.784995,4.826571,0,0.0,0,1.0,0,1,0,18,99999,99999,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
0,2,1.0,1.0,0,4.838775,4.784995,4.826571,0,0.0,0,1.0,0,1,0,47,99999,99999,1,0,0,0,0,0,0,0,1,1,0,0,0,0,1,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
